In [4]:
import numpy as np
from math import ceil

In [ ]:
w=[0.04, 0.20, 0.40, 0.28, 0.10, 0.16, 0.03, 0.08]
v=[0.10, 0.14, 0.24, 0.32, 0.28, 0.24, 0.18, 0.14]
q=[30, 20, 12, 23, 15, 30, 40, 25]
pi1=[1/10, 1/5, 1/2, 1/3, 1/3, 1/4, 1/5, 1/7]

In [11]:
a=[]
for i in range(8):
    a.append(min(int(1/w[i]), int(1/v[i])))
A=np.diag(a)
neg_I = -1 * np.eye(8)
A = np.hstack([A, neg_I])
A

array([[10.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -1., -0., -0., -0., -0.,
        -0., -0., -0.],
       [ 0.,  5.,  0.,  0.,  0.,  0.,  0.,  0., -0., -1., -0., -0., -0.,
        -0., -0., -0.],
       [ 0.,  0.,  2.,  0.,  0.,  0.,  0.,  0., -0., -0., -1., -0., -0.,
        -0., -0., -0.],
       [ 0.,  0.,  0.,  3.,  0.,  0.,  0.,  0., -0., -0., -0., -1., -0.,
        -0., -0., -0.],
       [ 0.,  0.,  0.,  0.,  3.,  0.,  0.,  0., -0., -0., -0., -0., -1.,
        -0., -0., -0.],
       [ 0.,  0.,  0.,  0.,  0.,  4.,  0.,  0., -0., -0., -0., -0., -0.,
        -1., -0., -0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  5.,  0., -0., -0., -0., -0., -0.,
        -0., -1., -0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  7., -0., -0., -0., -0., -0.,
        -0., -0., -1.]])

In [14]:
c1 = [1]*8 + [0]*8
B = A[:, :8]
cB=[1]*8
b1=np.linalg.inv(B)@q
A=np.linalg.inv(B)@A
sigma1=c1-cB@A
sigma1


array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.1       , 0.2       ,
       0.5       , 0.33333333, 0.33333333, 0.25      , 0.2       ,
       0.14285714])

In [3]:
wpi1=[w[i]/pi1[i] for i in range(8)]
wpi1

[0.39999999999999997,
 1.0,
 0.8,
 0.8400000000000001,
 0.30000000000000004,
 0.64,
 0.15,
 0.56]

In [4]:
vpi1=[v[i]/pi1[i] for i in range(8)]
vpi1

[1.0,
 0.7000000000000001,
 0.48,
 0.9600000000000001,
 0.8400000000000001,
 0.96,
 0.8999999999999999,
 0.9800000000000001]

找同时满足w和v约束代价最小的，依次为3、5、7、4、6、8、2、1，依次赋予可取的最大整数值，得到新的运输方案a9=[0,0,2,0,1,0,1,0]，代入上一步单纯形的列为B^(-1)*a9

In [17]:
a9=[0,0,2,0,1,0,1,0]
a9=np.linalg.inv(B)@a9
A=np.hstack([A, a9.reshape(-1,1)])
A

array([[ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        , -0.1       ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        , -0.2       ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        -0.5       ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  1.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        , -0.33333333,  0.        ,  0.        ,  0.        ,
         0.        

In [26]:
# c1=[1]*8+[0]*8
# c1.append(1)
sigma2=c1-cB@A
# print(sigma2) #证实了a9需进基
theta=b1/a9
print(theta)

[inf inf  6. inf 15. inf 40. inf]


/var/folders/_9/bwrp540d21j6tn6bbmtzrxww0000gn/T/ipykernel_59585/1276806201.py:5: RuntimeWarning: divide by zero encountered in divide
  theta=b1/a9


取最小theta对应的变量出基，即x_3

## 列生成法完整实现

**逻辑**：  
1. 用单纯形法求解当前有限主问题（RMP）得最优解与对偶 π。  
2. 用判别数（检验数）找改进列：解定价子问题，若存在 σ = c_new - π'a < 0 的列 a，则入基。  
3. 用最小比值 θ 选离基变量（θ = (x_B)_i / â_i，â = B⁻¹a）。  
4. 做一次主元（入基/离基），再对当前基用单纯形法求最优。  
5. 若定价子问题不再产生改进列（所有 σ ≥ 0），则停止。

In [27]:
def revised_simplex(A, b, c, basis, B_inv=None, x_B=None, max_iter=5000):
    """
    修订单纯形法：min c'x, Ax=b, x>=0.
    返回 (obj, basis, B_inv, x_B, pi) 或 (None, ...) 若无界/失败。
    """
    m, n = A.shape
    b = np.asarray(b).ravel().astype(float)
    c = np.asarray(c).ravel().astype(float)
    if B_inv is None:
        B_inv = np.linalg.inv(A[:, basis])
    if x_B is None:
        x_B = B_inv @ b
    pi = B_inv.T @ c[basis]
    for _ in range(max_iter):
        # 检验数：σ_j = c_j - π' A_j
        sigma = c - (A.T @ pi)
        # 非基变量索引
        non_basis = [j for j in range(n) if j not in basis]
        sigma_N = sigma[non_basis]
        if np.all(sigma_N >= -1e-10):
            obj = np.dot(c[basis], x_B)
            return obj, basis, B_inv, x_B, pi
        # 选最负检验数入基
        j_star = non_basis[np.argmin(sigma_N)]
        d = B_inv @ A[:, j_star]
        if np.all(d <= 1e-10):
            return None, basis, B_inv, x_B, pi  # 无界
        # 最小比值
        pos = d > 1e-10
        theta_arr = np.where(pos, x_B / d, np.inf)
        i_star = np.argmin(theta_arr)
        theta = theta_arr[i_star]
        if theta == np.inf:
            return None, basis, B_inv, x_B, pi
        # 主元：离基 basis[i_star]，入基 j_star
        l = i_star
        pivot = d[l]
        B_inv_new = B_inv.copy()
        B_inv_new[l, :] = B_inv[l, :] / pivot
        for i in range(m):
            if i != l:
                B_inv_new[i, :] = B_inv[i, :] - d[i] * B_inv_new[l, :]
        x_B_new = x_B - theta * d
        x_B_new[l] = theta
        basis_new = basis.copy()
        basis_new[l] = j_star
        basis, B_inv, x_B = basis_new, B_inv_new, x_B_new
        pi = B_inv.T @ c[basis]
    return None, basis, B_inv, x_B, pi  # 迭代超限

In [28]:
def pricing_subproblem(pi, w, v, a_max):
    """
    定价子问题：max π'a  s.t. w'a <= 1, v'a <= 1, 0 <= a <= a_max, a 整数.
    若最优值 > 1 则返回 (a, reduced_cost)；否则返回 (None, None).
    采用贪心：按 π_i 从大到小排序，依次取尽可能多的该物品（满足 w、v 约束）。
    """
    n = len(pi)
    a_max = list(a_max)
    # 按 max(w_i/π_i, v_i/π_i) 从小到大排，优先选“每单位收益所需资源负担”小的
    order = sorted(range(n), key=lambda i: max(w[i] / (pi[i] + 1e-12), v[i] / (pi[i] + 1e-12)))
    a = [0] * n
    w_used, v_used = 0.0, 0.0
    for i in order:
        max_add = a_max[i]
        if w[i] > 0:
            max_add = min(max_add, int((1 - w_used) / w[i]))
        if v[i] > 0:
            max_add = min(max_add, int((1 - v_used) / v[i]))
        max_add = max(0, max_add)
        a[i] = max_add
        w_used += a[i] * w[i]
        v_used += a[i] * v[i]
    val = sum(pi[i] * a[i] for i in range(n))
    if val > 1.0 + 1e-6:
        return np.array(a, dtype=float), 1.0 - val
    return None, None

In [29]:
def column_generation(A, b, c, basis, w, v, a_max, max_outer=5):
    """
    列生成主循环：
    1. 单纯形求 RMP 最优 → 得对偶 π
    2. 定价子问题求改进列 a（判别数 σ = c_new - π'a < 0）
    3. 若无改进列则停止；否则新列入基，最小 θ 选离基，主元一次
    4. 再对当前基做单纯形至最优，回到步骤 2
    """
    m, n = A.shape
    B_inv = np.linalg.inv(A[:, basis])
    x_B = B_inv @ b
    history = []
    for outer in range(max_outer):
        obj, basis, B_inv, x_B, pi = revised_simplex(
            A, b, c, basis, B_inv=B_inv, x_B=x_B
        )
        if obj is None:
            print("单纯形无界或失败")
            break
        history.append({"iter": outer, "obj": obj, "n_cols": A.shape[1]})
        a_new, red_cost = pricing_subproblem(pi, w, v, a_max)
        if a_new is None:
            print(f"第 {outer+1} 轮后无改进列，已最优。目标值 = {obj:.6f}")
            break
        # 新列入基：添加 a_new 到 A（原列是 8 维），c 增加 1
        a_col = a_new.reshape(-1, 1)
        A = np.hstack([A, a_col])
        c = np.append(c, 1.0)
        j_enter = A.shape[1] - 1
        d = B_inv @ A[:, j_enter]
        if np.all(d <= 1e-10):
            print("新列入基后问题无界")
            break
        theta_arr = np.where(d > 1e-10, x_B / d, np.inf)
        i_leave = int(np.argmin(theta_arr))
        theta = theta_arr[i_leave]
        if theta == np.inf:
            print("最小 θ 为无穷")
            break
        # 主元：离基 basis[i_leave]，入基 j_enter
        pivot = d[i_leave]
        B_inv_new = B_inv.copy()
        B_inv_new[i_leave, :] = B_inv[i_leave, :] / pivot
        for i in range(m):
            if i != i_leave:
                B_inv_new[i, :] = B_inv[i, :] - d[i] * B_inv_new[i_leave, :]
        x_B_new = x_B - theta * d
        x_B_new[i_leave] = theta
        basis_new = basis.copy()
        basis_new[i_leave] = j_enter
        basis, B_inv, x_B = basis_new, B_inv_new, x_B_new
        print(f"  添加新列，入基列 {j_enter}，离基行 {i_leave} (θ={theta:.4f})，当前列数 {A.shape[1]}")
    return A, b, c, basis, B_inv, x_B, history

In [30]:
# 用你已有的数据构建初始 RMP 并运行列生成
a_max_list = [min(int(1/w[i]), int(1/v[i])) for i in range(8)]
A0 = np.diag(a_max_list).astype(float)
A0 = np.hstack([A0, -np.eye(8)])
c0 = np.array([1.0] * 8 + [0.0] * 8)
b0 = np.array(q, dtype=float)
basis0 = list(range(8))

A_final, b_final, c_final, basis_final, B_inv_final, x_B_final, hist = column_generation(
    A0.copy(), b0, c0.copy(), basis0, w, v, a_max_list
)
print("\n最终基变量索引:", basis_final)
print("最终基变量取值 x_B:", x_B_final)
print("最终目标值:", np.dot(c_final[basis_final], x_B_final))

  添加新列，入基列 16，离基行 2 (θ=6.0000)，当前列数 17
  添加新列，入基列 17，离基行 4 (θ=3.0000)，当前列数 18
  添加新列，入基列 18，离基行 6 (θ=6.8000)，当前列数 19
第 4 轮后无改进列，已最优。目标值 = 40.429524

最终基变量索引: [0, 1, 16, 3, 17, 5, 18, 7]
最终基变量取值 x_B: [2.32       4.         6.         7.66666667 3.         7.5
 6.8        3.14285714]
最终目标值: 40.42952380952381


/var/folders/_9/bwrp540d21j6tn6bbmtzrxww0000gn/T/ipykernel_59585/1426057004.py:34: RuntimeWarning: divide by zero encountered in divide
  theta_arr = np.where(d > 1e-10, x_B / d, np.inf)


In [31]:
print(hist)

[{'iter': 0, 'obj': 44.738095238095234, 'n_cols': 16}, {'iter': 1, 'obj': 41.53809523809524, 'n_cols': 17}, {'iter': 2, 'obj': 41.10952380952381, 'n_cols': 18}, {'iter': 3, 'obj': 40.42952380952381, 'n_cols': 19}]


In [32]:
print(A_final)

[[10.  0.  0.  0.  0.  0.  0.  0. -1. -0. -0. -0. -0. -0. -0. -0.  0.  0.
   1.]
 [ 0.  5.  0.  0.  0.  0.  0.  0. -0. -1. -0. -0. -0. -0. -0. -0.  0.  0.
   0.]
 [ 0.  0.  2.  0.  0.  0.  0.  0. -0. -0. -1. -0. -0. -0. -0. -0.  2.  0.
   0.]
 [ 0.  0.  0.  3.  0.  0.  0.  0. -0. -0. -0. -1. -0. -0. -0. -0.  0.  0.
   0.]
 [ 0.  0.  0.  0.  3.  0.  0.  0. -0. -0. -0. -0. -1. -0. -0. -0.  1.  3.
   0.]
 [ 0.  0.  0.  0.  0.  4.  0.  0. -0. -0. -0. -0. -0. -1. -0. -0.  0.  0.
   0.]
 [ 0.  0.  0.  0.  0.  0.  5.  0. -0. -0. -0. -0. -0. -0. -1. -0.  1.  0.
   5.]
 [ 0.  0.  0.  0.  0.  0.  0.  7. -0. -0. -0. -0. -0. -0. -0. -1.  0.  1.
   0.]]


In [34]:
print(b_final)
print(c_final)
print(basis_final)
print(B_inv_final)




[30. 20. 12. 23. 15. 30. 40. 25.]
[1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1.]
[0, 1, 16, 3, 17, 5, 18, 7]
[[ 0.1         0.          0.01        0.          0.          0.
  -0.02        0.        ]
 [ 0.          0.2         0.          0.          0.          0.
   0.          0.        ]
 [ 0.          0.          0.5         0.          0.          0.
   0.          0.        ]
 [ 0.          0.          0.          0.33333333  0.          0.
   0.          0.        ]
 [ 0.          0.         -0.16666667  0.          0.33333333  0.
   0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.25
   0.          0.        ]
 [ 0.          0.         -0.1         0.          0.          0.
   0.2         0.        ]
 [ 0.          0.          0.02380952  0.         -0.04761905  0.
   0.          0.14285714]]


In [ ]:
# 根据basis_final和x_B_final，得到x_final
x_final=np.zeros(A_final.shape[1])
for i in range(len(basis_final)):
    x_final[basis_final[i]]=x_B_final[i]
print(x_final)

[2.32       4.         0.         7.66666667 0.         7.5
 0.         3.14285714 0.         0.         0.         0.
 0.         0.         0.         0.         6.         3.
 6.8       ]


In [37]:
# 将x_final向上取整，得到整数规划近似解
x_final_int=np.ceil(x_final)
print(x_final_int)

# 计算整数规划近似解的目标值
obj_int=np.dot(c_final, x_final_int)
print(obj_int)


[3. 4. 0. 8. 0. 8. 0. 4. 0. 0. 0. 0. 0. 0. 0. 0. 6. 3. 7.]
43.0
